# Train and evaluate on Colab

Fine-tunes one ImageNet-pretrained CNN on the NIH training patients, then runs the frozen model
over the NIH hold-out set and the external test sets. It is the same code as on the laptop
(`scripts/train.py`, `evaluate.py`, `summarise_results.py`, `compare_models.py`); this notebook
only moves files.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put in `MyDrive/malaria_thesis/`:
`code.zip` (or set `GITHUB_REPO`), `data_raw.zip` and, for RQ2, `data_rq2.zip`. All three come
from `python scripts/pack_for_colab.py`; each holds a `PROVENANCE_<zip>.txt` with the commit,
manifest hashes and (data zips) a digest of the crops it was packed from.

Checkpoints are written to Drive after every epoch, so if Colab disconnects, run all cells
again: training resumes where it stopped, and evaluation keeps the prediction files it already
wrote. To train several architectures in one session, change `ARCH` and rerun from the training
cell.

In [ ]:
ARCH = "vgg16"          # vgg16 | resnet50 | mobilenet_v2
SEED = 0
VARIANTS = ["raw", "masked", "reinhard", "histmatch"]   # ["raw"] if data_rq2.zip is not on Drive
FORCE_EVAL = False      # True: redo prediction files that already exist (e.g. after repacking)
GITHUB_REPO = ""        # e.g. "user/malaria-thesis"; empty = use code.zip from Drive
DRIVE_DIR = "/content/drive/MyDrive/malaria_thesis"
REPO = "/content/thesis"
ZIPS = ["data_raw.zip"] + (["data_rq2.zip"] if len(VARIANTS) > 1 else [])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code and data

In [ ]:
import os, subprocess
if not os.path.exists(REPO):
    if GITHUB_REPO:
        from google.colab import userdata   # add a GITHUB_TOKEN secret (key icon, left bar)
        token = userdata.get("GITHUB_TOKEN")
        subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_REPO}.git", REPO], check=True)
    else:
        subprocess.run(["unzip", "-q", f"{DRIVE_DIR}/code.zip", "-d", REPO], check=True)
os.chdir(REPO)
!ls

In [ ]:
# Copy each zip to the local disk first: unzipping 100k small files straight from Drive is slow.
# The marker stores the zip's size and date, so a re-uploaded zip is unpacked again, and it is
# written only after unzip succeeded (check=True raises otherwise).
import shutil
for z in ZIPS:
    src = f"{DRIVE_DIR}/{z}"
    st = os.stat(src)
    stamp = f"{st.st_size} {int(st.st_mtime)}"
    marker = f"{REPO}/.unzipped_{z}"
    if os.path.exists(marker) and open(marker).read() == stamp:
        continue
    shutil.copy(src, f"/content/{z}")
    subprocess.run(["unzip", "-q", "-o", f"/content/{z}", "-d", REPO], check=True)
    os.remove(f"/content/{z}")
    open(marker, "w").write(stamp)
!cat PROVENANCE_*.txt
!ls data/manifests

In [ ]:
# Checkpoints and predictions live on Drive, so they survive a disconnect.
for sub in ["models", "outputs/predictions"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
    os.makedirs(os.path.dirname(f"{REPO}/{sub}"), exist_ok=True)
    if not os.path.islink(f"{REPO}/{sub}"):
        !rm -rf "{REPO}/{sub}"
        os.symlink(f"{DRIVE_DIR}/{sub}", f"{REPO}/{sub}")
# Colab already ships torch, torchvision, OpenCV, scikit-image, pandas and scikit-learn;
# requirements.txt is not installed here because its pinned numpy can break Colab's torch.
!python -c "import torch, torchvision, cv2, skimage; print(torch.__version__, torchvision.__version__)"

## Check, train, evaluate

In [ ]:
# Smoke check on 64 cells; written to the local disk, not Drive (checkpoints are large).
!python scripts/train.py --arch {ARCH} --smoke --workers 2 --out /content/smoke

In [ ]:
!python scripts/train.py --arch {ARCH} --seed {SEED} --workers 2 --resume

In [ ]:
!python scripts/evaluate.py --checkpoint models/{ARCH}_s{SEED}.pt --variants {" ".join(VARIANTS)} --workers 2 {"--force" if FORCE_EVAL else ""}

In [ ]:
TAG = f"{ARCH}_s{SEED}"
!python scripts/summarise_results.py --model {TAG} --variants {" ".join(VARIANTS)}
!mkdir -p "{DRIVE_DIR}/outputs/tables" "{DRIVE_DIR}/outputs/figures"
!cp outputs/tables/rq*_{TAG}*.csv "{DRIVE_DIR}/outputs/tables/"
!cp outputs/figures/R_{TAG}.png "{DRIVE_DIR}/outputs/figures/"
!cp models/{TAG}_log.json "{DRIVE_DIR}/outputs/tables/"
!cp outputs/predictions/{TAG}/run.json "{DRIVE_DIR}/outputs/tables/{TAG}_run.json"
!ls "{DRIVE_DIR}/outputs/tables"

## RQ1: compare the architectures

Once all three models are evaluated (raw variant at least). Every pair gets the difference in
drop with a paired 95% interval; the script stops if the models were scored on different
manifests or cells.

In [ ]:
!python scripts/compare_models.py --models vgg16_s{SEED} resnet50_s{SEED} mobilenet_v2_s{SEED} --name s{SEED}
!cp outputs/tables/rq1_compare_s{SEED}.csv "{DRIVE_DIR}/outputs/tables/"
!cp outputs/figures/R_compare_s{SEED}.png "{DRIVE_DIR}/outputs/figures/"

In [ ]:
from IPython.display import Image
Image(f"outputs/figures/R_{ARCH}_s{SEED}.png")